# BirdCLEF 2026 — Model_5 + HGNet 4-backbone ONNX ensemble (v22)

HGNet 分支 = 4 个 backbone 的 ONNX ensemble, 以 **24%** 权重 rank-blend 进 `subm_5.csv`:

| backbone | dir | 内部权重 | 折算最终占比 |
|---|---|---|---|
| hgnetv2_b0 | `hgnetv2_b0_onnx` | 3 (0.125) | **3%** |
| hgnetv2_b1 | `hgnetv2_b1_onnx` | 3 (0.125) | **3%** |
| hgnetv2_b4 (weak labels) | `hgnetv2_b4_onnx_weak_labels` | 15 (0.625) | **15%** |
| repvit_m1_1 | `repvit_m_1_1_onnx` | 3 (0.125) | **3%** |

`HGNET_W=0.24`, 内部权重 `[3,3,15,3]`. mel 每个文件只算一次, 喂给全部 4×4=16 个 ONNX session.

## Required Datasets to Attach

| Dataset | Why |
|---|---|
| `birdclef-2026` (competition) | hidden test audio + sample_submission |
| 你的 **v22 code dataset** (含 `scheme_model_5/` + 顶层 `hgnet_branch.py`) | Model_5 主代码 + ensemble 逻辑 |
| 你的 **hgnetv2_b0 ONNX** (含 `best_model_fold0..3.onnx`) | HGNet b0 4-fold |
| 你的 **hgnetv2_b1 ONNX** (含 `best_model_fold0..3.onnx`) | HGNet b1 4-fold |
| 你的 **hgnetv2_b4 weak-labels ONNX** (含 `best_model_fold0..3.onnx`) | HGNet b4 4-fold |
| `rishikeshjani/perch-onnx-for-birdclef-2026` | Perch ONNX |
| `google/bird-vocalization-classifier` (`tensorflow2/perch_v2_cpu/1`) | Perch TF fallback |
| `ashok205/tf-wheels` | offline TF wheel (Internet OFF) |
| `tuckerarrants/bc2026-distilled-sed-public` | Distilled-SED `sed_fold*.onnx` |

**Settings**: Internet **OFF**, Accelerator: **None** (CPU). Save Version → **Save & Run All** → Submit.

In [ ]:
import os, sys, glob

# =============================================================================
# 0. 用户必填 — 按 Kaggle 右侧 Input 列表里的实际路径改
# =============================================================================
# 0a. v22 code dataset 顶层 (含 scheme_model_5/ + hgnet_branch.py)
CODE_DIR  = '/kaggle/input/scheme-model-v22-hgnetv2-3-ensemble'

# 0b. HGNet 3 个 backbone 的 ONNX 目录 (各含 best_model_fold0..3.onnx)
#     ★ 手动填: 从 Kaggle 右侧 Input 列表复制实际路径, 顺序 = [b0, b1, b4_weak]
#       要跟下面 HGNET_WEIGHTS 一一对应. 路径填错会直接报错 (不做自动查找).
HGNET_DIRS = [
    '/kaggle/input/hgnetv2-b0-onnx',
    '/kaggle/input/hgnetv2-b1-onnx',
    '/kaggle/input/hgnetv2-b4-onnx-weak-labels',
]
# 跨 backbone 内部权重 (会被归一化): [1,1,3] -> 0.2 / 0.2 / 0.6
HGNET_WEIGHTS = '1,1,3'
# HGNet ensemble 在最终 rank-blend 中的总权重 (0.20 = 20%)
HGNET_W   = '0.20'

# 0c. SED ONNX 目录 (含 sed_fold*.onnx). ★ 手动填, 必填, 路径错直接报错.
SED_DIR   = '/kaggle/input/bc2026-distilled-sed-public'

print('CODE_DIR    =', CODE_DIR)
print('HGNET_DIRS  =', HGNET_DIRS)
print('HGNET_W     =', HGNET_W, ' weights =', HGNET_WEIGHTS)
print('SED_DIR     =', SED_DIR)

In [ ]:
# =============================================================================
# 1. 校验 HGNet 路径 — 完全用上面手填的 HGNET_DIRS, 路径不对直接报错
#    (不做 rglob 自动兜底, 请确保 0b 里的 3 个路径填对)
# =============================================================================
for d in HGNET_DIRS:
    assert os.path.isdir(d), f'HGNET_DIRS 路径不存在: {d}  (请改 cell 0b)'
    n = len(glob.glob(os.path.join(d, 'best_model_fold*.onnx')))
    print(f'>>> {d}  ({n} folds)')
    assert n >= 1, f'{d} 下没有 best_model_fold*.onnx  (请改 cell 0b)'

# SED 路径校验 (手动填, 不自动查找)
assert SED_DIR, 'SED_DIR 为空, 请在 cell 0c 手动填 SED ONNX 目录'
assert os.path.isdir(SED_DIR), f'SED_DIR 路径不存在: {SED_DIR}  (请改 cell 0c)'
_sed_n = len(glob.glob(os.path.join(SED_DIR, 'sed_fold*.onnx')))
print(f'>>> SED_DIR {SED_DIR}  ({_sed_n} folds)')
assert os.path.isfile(os.path.join(SED_DIR, 'sed_fold0.onnx')), \
    f'{SED_DIR} 下没有 sed_fold0.onnx  (请改 cell 0c)'

# code dir 兜底
if not os.path.isdir(os.path.join(CODE_DIR, 'scheme_model_5')):
    _hit = sorted(glob.glob('/kaggle/input/**/scheme_model_5/main.py', recursive=True))
    assert _hit, 'scheme_model_5/main.py 找不到, 请 attach v22 code dataset 并改 CODE_DIR'
    CODE_DIR = os.path.dirname(os.path.dirname(_hit[0]))
    print(f'!!! 自动选用 CODE_DIR = {CODE_DIR}')
assert os.path.isfile(os.path.join(CODE_DIR, 'hgnet_branch.py')), \
    f'{CODE_DIR} 下没有 hgnet_branch.py'
print('>>> CODE_DIR =', CODE_DIR)

In [ ]:
# =============================================================================
# 2. !!! 关键: 必须在 import scheme_model_5 之前设 env vars !!!
#    config.py 是模块级 os.environ.get, import 时一次性读完, 之后改无效.
# =============================================================================
os.environ['BIRDCLEF_MODEL5_SUBM']   = '/kaggle/working/submission.csv'
os.environ['BIRDCLEF_MODE']          = 'submit'
os.environ['BIRDCLEF_SED_DIR']       = SED_DIR

# HGNet 多 backbone ONNX ensemble 分支
os.environ['BIRDCLEF_M5_USE_HGNET']        = '1'
os.environ['BIRDCLEF_HGNET_W']             = HGNET_W
# Linux/Kaggle os.pathsep = ':' ; config 用 os.pathsep split
os.environ['BIRDCLEF_HGNET_ENSEMBLE_DIRS'] = os.pathsep.join(HGNET_DIRS)
os.environ['BIRDCLEF_HGNET_ENSEMBLE_WEIGHTS'] = HGNET_WEIGHTS
os.environ['BIRDCLEF_HGNET_BRANCH_PATH']   = os.path.join(CODE_DIR, 'hgnet_branch.py')

print('>>> env vars set (before import scheme_model_5):')
for _k in ['BIRDCLEF_MODEL5_SUBM', 'BIRDCLEF_M5_USE_HGNET', 'BIRDCLEF_HGNET_W',
           'BIRDCLEF_HGNET_ENSEMBLE_DIRS', 'BIRDCLEF_HGNET_ENSEMBLE_WEIGHTS',
           'BIRDCLEF_HGNET_BRANCH_PATH']:
    print(f'      {_k} = {os.environ.get(_k)!r}')

In [ ]:
# =============================================================================
# 3. sys.path + 清旧缓存 + import (此刻 config 才被初始化, 读到上面设的 env)
# =============================================================================
sys.path = [CODE_DIR] + [p for p in sys.path if p != CODE_DIR]
for _k in [k for k in list(sys.modules.keys()) if k.startswith('scheme_model_')]:
    del sys.modules[_k]

import scheme_model_5.main as _sm5_main
from scheme_model_5.config import (
    USE_HGNET as _USE_HGNET,
    HGNET_W as _HGNET_W,
    HGNET_ENSEMBLE_DIRS as _ENS_DIRS,
    HGNET_ENSEMBLE_WEIGHTS as _ENS_W,
    OUTPUT_SUBM as _OUTPUT_SUBM,
)
print(f'>>> import 实际加载       = {_sm5_main.__file__}')
print(f'>>> config.USE_HGNET      = {_USE_HGNET}')
print(f'>>> config.HGNET_W        = {_HGNET_W}')
print(f'>>> config.ENSEMBLE_DIRS  = {_ENS_DIRS}')
print(f'>>> config.ENSEMBLE_W     = {_ENS_W}')
print(f'>>> config.OUTPUT_SUBM    = {_OUTPUT_SUBM}')
assert _USE_HGNET, 'USE_HGNET 还是 False! 检查 BIRDCLEF_M5_USE_HGNET.'
assert len(_ENS_DIRS) == 3, f'ensemble dirs 应为 3 个, 实际 {len(_ENS_DIRS)}'

os.chdir('/kaggle/working')

In [ ]:
# =============================================================================
# 4. run Model_5 (主流水线 + HGNet 3-backbone ensemble rank-blend 20%)
# =============================================================================
_sm5_main.main()

In [ ]:
# =============================================================================
# 5. sanity check
# =============================================================================
import pandas as pd, shutil
_final_csv = '/kaggle/working/submission.csv'
if not os.path.exists(_final_csv):
    _alt = '/kaggle/working/subm_5.csv'
    if os.path.exists(_alt):
        shutil.copy(_alt, _final_csv)
        print(f'>>> 从 {_alt} 复制到 {_final_csv}')
df = pd.read_csv(_final_csv)
print('submission.csv shape:', df.shape)
print('row_id sample :', df['row_id'].head(3).tolist())
print('value range  :', float(df.iloc[:, 1:].min().min()), '..', float(df.iloc[:, 1:].max().max()))
assert df['row_id'].is_unique, 'duplicate row_id!'
assert df.iloc[:, 1:].notna().all().all(), 'NaN in submission!'
df.head(2)